# Gestion des Ruptures de Stock — Prévision & Alertes
## Étape 1 : Exploration des données

Dans ce notebook, j'utilise Pandas pour explorer l'historique de la demande produit. L'objectif est de comprendre la structure des données, d'identifier les produits les plus demandés et de repérer ceux qui ont une variabilité si forte qu'ils risquent la rupture de stock.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration visuelle
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

data_path = "../data/Historical Product Demand.csv"

print("Chargement des données...")
df = pd.read_csv(data_path)

print(f"Dimensions du dataset : {df.shape}")
print("\nTypes de colonnes :\n", df.dtypes)

### 1. Nettoyage et Formatage des Données
La colonne `Order_Demand` contient parfois des caractères spéciaux (comme des parenthèses) qu'il faut nettoyer. La colonne `Date` doit être au format datetime.

In [ ]:
# Nettoyage de la demande (retrait des parenthèses et conversion en numérique)
df['Order_Demand'] = df['Order_Demand'].astype(str).str.replace(r'[^0-9-]', '', regex=True)
df['Order_Demand'] = pd.to_numeric(df['Order_Demand'], errors='coerce')

# Nettoyage des dates
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Suppression des lignes avec des valeurs manquantes critiques
df = df.dropna(subset=['Date', 'Order_Demand'])

print("Aperçu après nettoyage :")
print(df.head())

### 2. Analyse de la Demande par Produit et Entrepôt
Quels sont les entrepôts les plus sollicités et les produits les plus demandés ?

In [ ]:
# Demande par entrepôt
warehouse_demand = df.groupby('Warehouse')['Order_Demand'].sum().sort_values(ascending=False)
print("Demande totale par entrepôt :\n", warehouse_demand)

# Top 10 produits
top_products = df.groupby('Product_Code')['Order_Demand'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_products.values, y=top_products.index, palette="Blues_r")
plt.title("Top 10 des Produits les plus demandés")
plt.xlabel("Volume Total Demandé")
plt.ylabel("Code Produit")
plt.show()

### 3. Détection des Produits à Risque de Rupture
Un produit est à risque s'il a un volume de demande élevé, mais surtout une **très forte variabilité** (écart-type élevé par rapport à la moyenne). J'utilise le Coefficient de Variation (CV).

In [ ]:
# Agrégation mensuelle par produit pour calculer la variabilité
df['YearMonth'] = df['Date'].dt.to_period('M')
monthly_demand = df.groupby(['Product_Code', 'YearMonth'])['Order_Demand'].sum().reset_index()

product_stats = monthly_demand.groupby('Product_Code')['Order_Demand'].agg(['mean', 'std', 'sum']).dropna()
product_stats['CV'] = product_stats['std'] / product_stats['mean']  # Coefficient de Variation

# On filtre les produits ayant un gros volume de ventes (top 5%)
volume_threshold = product_stats['sum'].quantile(0.95)
high_volume_products = product_stats[product_stats['sum'] >= volume_threshold]

# Les produits à risque sont ceux avec un fort volume et un CV élevé
risky_products = high_volume_products.sort_values(by='CV', ascending=False).head(10)

print("Top 10 Produits à Risque de Rupture (Volume Élevé + Forte Variabilité) :")
print(risky_products[['sum', 'mean', 'CV']])

### 4. Visualisation de la Saisonnalité
Je regarde l'évolution globale de la demande pour détecter des pics saisonniers.

In [ ]:
global_timeline = df.groupby('YearMonth')['Order_Demand'].sum()

plt.figure(figsize=(14, 5))
global_timeline.plot(marker='o', linestyle='-', color='#e74c3c')
plt.title("Évolution Globale Mensuelle de la Demande")
plt.xlabel("Mois")
plt.ylabel("Demande Totale")
plt.grid(True)
plt.show()

## Conclusions Business

1. **Concentration des volumes** : L'entrepôt `Whse_J` gère la grande majorité de la demande. Toute rupture ou problème logistique dans cet entrepôt impactera massivement l'entreprise.
2. **Loi de Pareto** : Une petite poignée de produits (comme le `Product_1359`) représente une part colossale de la demande totale. Il faut des stocks de sécurité bétonnés sur ce top 10.
3. **Danger de l'imprévisibilité** : Mon calcul du coefficient de variation (CV) a mis en évidence des produits à forte rotation dont la demande fluctue énormément d'un mois à l'autre. Ces produits sont nos principaux candidats aux ruptures de stock car un réapprovisionnement constant basé sur une moyenne simple échouera lamentablement.